In [29]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
import random
warnings.filterwarnings("ignore")

In [30]:
def second_degree(p):
    return 1.1*p - 0.5*p**2


def fourth_degree(p):
    return -p**4 + 22*p**3 - 165*p**2 + 480*p-150


def exponential(p):
    return 100*np.exp((-((p-5) ** 2))/20)

In [43]:
# Create a dictionary mapping strings to functions
function_mapping = {
    "second": second_degree,
    "fourth": fourth_degree,
    "exponential": exponential
}

degree_mapping = {
    "second": 2,
    "fourth": 4,
    'exponential': 4
}

bounds_mapping = {
    "second": [0., 2.2],
    "fourth": [0.36, 10.5],
    "exponential": [0.0, 10.0]
}

noise_mapping = {
    "second": 0.1,
    "fourth": 10,
    "exponential": 3
}

In [44]:
from scipy.optimize import minimize_scalar

# Define the negative of the fourth_degree function to find the maximum


def neg_fourth_degree(p):
    return -fourth_degree(p)


# Use minimize_scalar to find the value of p that minimizes the negative fourth_degree function
result = minimize_scalar(neg_fourth_degree, bounds=(0, 10), method='bounded')

# The optimal value of p
optimal_p_fourth = result.x
optimal_p_second = 1.1
optimal_p_exponent = 5

In [45]:
SIZE = 100

actual_revenue_function="fourth"
model_revenue_function="fourth"

In [46]:
if actual_revenue_function == "second":
    optimal = np.array([second_degree(optimal_p_second) for _ in range(SIZE)])
elif actual_revenue_function == "fourth":
    optimal = np.array([fourth_degree(optimal_p_fourth) for _ in range(SIZE)])
elif actual_revenue_function == "exponential":
    optimal = np.array([exponential(optimal_p_exponent) for _ in range(SIZE)])

In [47]:
def Iterated_Least_Squares(actual_revenue_function="fourth",model_revenue_function="fourth"):
    data = []
    final = []
    # np.random.seed(np.random.randint(0, 1000))
    p = np.random.uniform(
        bounds_mapping[model_revenue_function][0], bounds_mapping[model_revenue_function][1])
    y = function_mapping[actual_revenue_function](
        p)+np.random.normal(0, noise_mapping[actual_revenue_function])
    data.append([p, y])
    final.append(function_mapping[actual_revenue_function](p))
    for i in (range(1, SIZE)):
        df = pd.DataFrame(data, columns=['p', 'y'])
        coeffs = np.polyfit(
            df['p'], df['y'], degree_mapping[model_revenue_function])
        # print(coeffs)
        # coeffs = gradient_descent(df['p'], df['y'], degree_mapping[model_revenue_function])
        # Define the polynomial function
        
        poly = np.poly1d(coeffs)

        # Find the value of x that maximizes the polynomial
        # Find the critical points by finding the roots of the derivative of the polynomial
        poly_derivative = np.polyder(poly)
        critical_points = np.roots(poly_derivative)

        # critical_points = np.append(
        #     critical_points, bounds_mapping[model_revenue_function])

        # Evaluate the polynomial at the critical points and the boundaries
        critical_points = np.append(critical_points, [0, 10])
        critical_points = critical_points[np.isreal(critical_points)]
        critical_points = critical_points[(critical_points >= 0) & (critical_points <= 10)]
        x = np.real(critical_points)
        # x = np.clip(x, 0, None)
        y = np.real(poly(x))
        # print(x)
        final.append(function_mapping[actual_revenue_function](
            x[np.argmax(y)]))
        data.append([x[np.argmax(y)], function_mapping[actual_revenue_function](
            x[np.argmax(y)]) + np.random.normal(0, noise_mapping[actual_revenue_function])])
        # print("coeffs:", coeffs)
        # print("revenue:", final[-1])
        # print("price:", x[np.argmax(y)])
        # print("difference", optimal[0]-final[-1])
    return final

In [48]:
results=Iterated_Least_Squares()

In [59]:
import json

data_total=[]
for actual_revenue_function in tqdm(["second", "fourth", "exponential"]):
    for model_revenue_function in ["second", "fourth", "exponential"]:
        data = {
                "actual_dim": actual_revenue_function,
                "model_dim": model_revenue_function,
                "revenue": [],
                "regret": []
            }
        
        for i in range(100):
            rets = Iterated_Least_Squares(actual_revenue_function=actual_revenue_function, model_revenue_function=model_revenue_function)
            # print(rets)
            if actual_revenue_function == "second":
                optimal = np.array([second_degree(optimal_p_second) for _ in range(SIZE)])
            elif actual_revenue_function == "fourth":
                optimal = np.array([fourth_degree(optimal_p_fourth) for _ in range(SIZE)])
            elif actual_revenue_function == "exponential":
                optimal = np.array([exponential(optimal_p_exponent) for _ in range(SIZE)])
                
            cumulative_diff = (optimal - np.array([item for item in rets])).tolist()
            data["revenue"].append(rets)
            data["regret"].append(cumulative_diff)
            
        data_total.append(data)
with open('data.json', 'w') as f:
    json.dump(data_total, f)


100%|██████████| 3/3 [00:41<00:00, 13.92s/it]


In [55]:
def f_fun(coeff_array,*args):
  sum = 0
  p_arr,d_arr = args
  for i in range(len(p_arr)):
    p = p_arr[i]
    p_vec = np.array([1,p,p**2])
    demo_demand = np.sum(coeff_array*p_vec)
    # demo_demand = func_revenue(p)
    sum += (d_arr[i]-demo_demand)**2
  # print(sum)
  return sum

In [56]:
def g_fun(p,*args):
  coeff_array = args
  p=p[0]
  p_vec = np.array([1,p,p**2])
  # print(-np.sum(p_vec*coeff_array))
  return -np.sum(p_vec*coeff_array)

In [57]:
from scipy.optimize import minimize_scalar
from scipy.optimize import minimize

# Define the negative of the fourth_degree function to find the maximum
def neg_fourth_degree(p):
    return -fourth_degree(p)

# Use minimize_scalar to find the value of p that minimizes the negative fourth_degree function
result = minimize_scalar(neg_fourth_degree, bounds=(0, 10), method='bounded')

# The optimal value of p
optimal_p_fourth = result.x

In [58]:
def GreedyILS():
  p_demo = np.random.uniform(low = 0,high = 10,size = 1)[0]
  d_demo = fourth_degree(p_demo)
  p_array = []
  d_array = []
  p_array.append(p_demo)
  d_array.append(d_demo)
  epsilon_array = []
  for t in tqdm(range(SIZE-1)):
    coeff_estimate = minimize(f_fun,x0 =np.random.random_sample(size = (3,)),args = (p_array,d_array),bounds = [(-50.,50.),(-50.,50.),(-50.,50.)] )["x"]
    # print(coeff_estimate)
    coeff_estimate = np.array(coeff_estimate)
    # print(coeff_estimate)
    price_estimate = minimize(g_fun,x0 = np.array([random.random()]),args = (coeff_estimate),bounds = [(0,10)])["x"][0]
    p_array.append(price_estimate)
    d_new = fourth_degree(price_estimate)+np.random.normal(0,10)
    d_array.append(d_new)
    # reven_array = np.array(p_array)*np.array(d_array)
    # epsilon =np.max([0,np.min(100 - np.array(d_array))])
    # epsilon_array.append(epsilon)
  return np.array(epsilon_array),d_array